In [1]:
import spacy
import pandas as pd

#spacy.cli.download("en_core_web_sm")

from pathlib import Path
from datasets import load_dataset

from src.lexicon_builder import *
from src.corpus_adaptors import *
from src.extractors import *

## Common Crawl

In [ ]:
nlp = spacy.load("en_core_web_sm") 

In [ ]:
CC_dataset = load_dataset('codymd/cc100_en_sample')
CC_dataset_text = CC_dataset['train']['text']
cutoff = 100_000

### Unigrams

In [ ]:
CCfile = Path(f'/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/CommonCrawl/CC_unigram_lexicon_{cutoff}.csv')

uni_gram_dep = load_or_build_lexicon(
    out_path=CCfile,
    builder_fn=lambda: build_lexicon(
        texts=iter_texts_from_dataset(CC_dataset_text, limit_docs=cutoff),
        nlp=nlp,
        extractor=extract_unigrams_from_doc,
        batch_size=100,
        n_process=1,
        key_names=("lemma","pos"),
    ),
    key_names=("lemma","pos"),
)

### Bigrams

In [ ]:
CCfile = Path(f'/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/CommonCrawl/CC_bigram_lexicon_{cutoff}.csv')

uni_gram_dep = load_or_build_lexicon(
    out_path=CCfile,
    builder_fn=lambda: build_lexicon(
        texts=iter_texts_from_dataset(CC_dataset_text, limit_docs=cutoff),
        nlp=nlp,
        extractor=extract_bigrams_from_doc,
        batch_size=100,
        n_process=1,
        key_names=("lemma1","lemma2"),
    ),
    key_names=("lemma1","lemma2"),
)

### Constrained bigrams

In [ ]:
CCfile = Path(f'/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/CommonCrawl/CC_dep_lexicon_{cutoff}.csv')

uni_gram_dep = load_or_build_lexicon(
    out_path=CCfile,
    builder_fn=lambda: build_lexicon(
        texts=iter_texts_from_dataset(CC_dataset_text, limit_docs=cutoff),
        nlp=nlp,
        extractor=extract_dep_pairs_from_doc,
        batch_size=100,
        n_process=1,
        key_names=("lemma1","lemma2"),
    ),
    key_names=("lemma1","lemma2"),
)

## PeopleSpeech

In [ ]:
PS_dataset = load_dataset('MLCommons/peoples_speech', 'test')
PS_dataset_text = PS_dataset['test']['text']

In [ ]:
out_paths = {
    1: Path(f"PeopleSpeech/PS_pos_unigram_{cutoff}.csv"),
    2: Path(f"PeopleSpeech/PS_pos_bigram_{cutoff}.csv"),
    3: Path(f"PeopleSpeech/PS_pos_trigram_{cutoff}.csv"),
}

dfs = load_or_build_pos_ngram_lexicons(
    out_paths=out_paths,
    texts_factory=lambda: iter_texts_from_dataset(PS_dataset_text, limit_docs=cutoff),
    nlp=nlp,
    orders=(1,2,3),
    batch_size=100,
    n_process=1,
)

df_uni, df_bi, df_tri = dfs[1], dfs[2], dfs[3]

## BasiScript

In [2]:
nlp = spacy.load("nl_core_news_lg") 

### Unigrams

In [ ]:
BSfile = Path(f'/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/BasiScript/BS_unigram_lexicon.csv')
BS_data_path = Path('/Users/sabijn/Documents/PhD/Datasets/BS_all')

uni_gram_dep = load_or_build_lexicon(
    out_path=BSfile,
    builder_fn=lambda: build_lexicon(
        texts=iter_texts_from_folia_xml(BS_data_path),
        nlp=nlp,
        extractor=extract_unigrams_from_doc,
        batch_size=100,
        n_process=1,
        key_names=("lemma","pos"),
    ),
    key_names=("lemma","pos"),
)

### Bigrams

In [ ]:
BSfile = Path(f'/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/BasiScript/BS_bigram_lexicon.csv')
BS_data_path = Path('/Users/sabijn/Documents/PhD/Datasets/BS_all')

uni_gram_dep = load_or_build_lexicon(
    out_path=BSfile,
    builder_fn=lambda: build_lexicon(
        texts=iter_texts_from_folia_xml(BS_data_path),
        nlp=nlp,
        extractor=extract_bigrams_from_doc,
        batch_size=100,
        n_process=1,
        key_names=("lemma1","lemma2"),
    ),
    key_names=("lemma1","lemma2"),
)

### Constrained bigrams

In [3]:
BSfile = Path(f'/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/BasiScript/BS_dep_lexicon_newlemmatizer_nopronoun.csv')
BS_data_path = Path('/Users/sabijn/Documents/PhD/Datasets/BS_all')

uni_gram_dep = load_or_build_lexicon(
    out_path=BSfile,
    builder_fn=lambda: build_lexicon(
        texts=iter_texts_from_folia_xml(BS_data_path),
        nlp=nlp,
        extractor=extract_dep_pairs_from_doc,
        batch_size=100,
        n_process=1,
        key_names=("lemma1","lemma2"),
    ),
    key_names=("lemma1","lemma2"),
)

Building lexicon: 0it [00:00, ?it/s]

In [4]:
def maplex2unk(path_name, unk_token, cutoff):
    lexicon_raw = pd.read_csv(path_name)
    if lexicon_raw.empty:
        raise ValueError("Training lexicon is empty — cannot compute perplexity.")
    
    lex = lexicon_raw.copy()
    lex['freq'] = lex['freq'].astype(int)

    # Identify rare lemma2 and replace them with <unk>
    lemma2_freq = lex.groupby('lemma2')['freq'].sum()
    rare_lemma2 = set(lemma2_freq[lemma2_freq < cutoff].index)
    lex['lemma2'] = lex['lemma2'].apply(lambda x: unk_token if x in rare_lemma2 else x)

    # Aggregate after replacement (some bigrams will merge)
    lex = lex.groupby(['lemma1', 'lemma2'], as_index=False)['freq'].sum()
    lex.to_csv('/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/BasiScript/BS_dep_lexicon_unk_newlemmatizer_nopronoun.csv')


In [5]:
maplex2unk(f'/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/BasiScript/BS_dep_lexicon_newlemmatizer_nopronoun.csv', '<unk>', 2)

### POS corpus

In [2]:
BS_data_path = Path('/Users/sabijn/Documents/PhD/Datasets/BS_all')
BS_list = iter_texts_from_folia_xml(BS_data_path)

In [6]:
out_paths = {
    1: Path(f"BasiScript/BasiScript_pos_unigram.csv"),
    2: Path(f"BasiScript/BasiScript_pos_bigram.csv"),
    3: Path(f"BasiScript/BasiScript_pos_trigram.csv"),
}

dfs = load_or_build_pos_ngram_lexicons(
    out_paths=out_paths,
    texts_factory=lambda: iter_texts_from_dataset(BS_list),
    nlp=nlp,
    orders=(1,2,3),
    batch_size=100,
    n_process=1,
)

Building POS n-grams: 0it [00:00, ?it/s]

## CGN (Corpus Gesproken Nederlands)

In [ ]:
CGN_dataset = pd.read_csv('CGN/CGN_text.csv')
CGN_dataset_text = CGN_dataset['text'].to_list()

In [ ]:
out_paths = {
    1: Path(f"CGN/CGN_pos_unigram.csv"),
    2: Path(f"CGN/CGN_pos_bigram.csv"),
    3: Path(f"CGN/CGN_pos_trigram.csv"),
}

dfs = load_or_build_pos_ngram_lexicons(
    out_paths=out_paths,
    texts_factory=lambda: iter_texts_from_dataset(CGN_dataset_text),
    nlp=nlp,
    orders=(1,2,3),
    batch_size=100,
    n_process=1,
)

df_uni, df_bi, df_tri = dfs[1], dfs[2], dfs[3]

## Basilex

In [ ]:
import warnings
warnings.filterwarnings("ignore")

nlp = spacy.load("nl_core_news_lg") 

In [ ]:
BLfile = Path(f'/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/BasiLex/BL_dep_lexicon.csv')
BL_data_path = Path('/Users/sabijn/Documents/PhD/Datasets/BasiLex-corpus/Data')

uni_gram_dep = load_or_build_lexicon(
    out_path=BLfile,
    builder_fn=lambda: build_lexicon(
        texts=iter_texts_from_folia_xml(BL_data_path),
        nlp=nlp,
        extractor=extract_dep_pairs_from_doc,
        batch_size=100,
        n_process=1,
        key_names=("lemma1","lemma2"),
    ),
    key_names=("lemma1","lemma2"),
)

## Wikipedia (Dutch)

In [11]:
wiki_dutch = load_dataset("wikimedia/wikipedia", "20231101.nl", split='train[:1%]')

README.md: 0.00B [00:00, ?B/s]

20231101.nl/train-00000-of-00006.parquet:   0%|          | 0.00/506M [00:00<?, ?B/s]

20231101.nl/train-00001-of-00006.parquet:   0%|          | 0.00/304M [00:00<?, ?B/s]

20231101.nl/train-00002-of-00006.parquet:   0%|          | 0.00/95.7M [00:00<?, ?B/s]

20231101.nl/train-00003-of-00006.parquet:   0%|          | 0.00/57.8M [00:00<?, ?B/s]

20231101.nl/train-00004-of-00006.parquet:   0%|          | 0.00/109M [00:00<?, ?B/s]

20231101.nl/train-00005-of-00006.parquet:   0%|          | 0.00/364M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2135977 [00:00<?, ? examples/s]

In [15]:
wiki_dataset_text = wiki_dutch['text']

In [16]:
Wikifile = Path(f'/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/WikiDutch/WD_dep_lexicon.csv')

uni_gram_dep = load_or_build_lexicon(
    out_path=Wikifile,
    builder_fn=lambda: build_lexicon(
        texts=iter_texts_from_dataset(wiki_dataset_text),
        nlp=nlp,
        extractor=extract_dep_pairs_from_doc,
        batch_size=100,
        n_process=1,
        key_names=("lemma1","lemma2"),
    ),
    key_names=("lemma1","lemma2"),
)

Building lexicon: 0it [00:00, ?it/s]

In [17]:
nlp_en = spacy.load("en_core_web_sm") 
nlp_dutch = spacy.load("nl_core_news_lg") 

In [29]:
en_text_newlines = "I am going to work today.\n I like my work."
en_text_no_newlines = "I am going to work today. I like my work."
nl_text_newlines = "Ik ga naar werk vandaag.\n Ik houd van mijn werk."
nl_text_no_newlines = "Ik ga naar werk vandaag. Ik houd van mijn werk."

In [26]:
sents_en = len(list(nlp_en(en_text_newlines).sents))
sents_en_no = len(list(nlp_en(en_text_no_newlines).sents))
sents_nl = list(nlp_dutch(nl_text_newlines).sents)
sents_nl_no = len(list(nlp_dutch(nl_text_no_newlines).sents))

In [27]:
sents_en, sents_en_no, len(sents_nl), sents_nl_no

(2, 9, 3, 2)

In [28]:
sents_nl

[Ik ga naar werk vandaag.,
 
  ,
 Ik houd van mijn werk.]